In [ ]:
! pip install -q transformers datasets evaluate accelerate huggingface_hub sacrebleu

- Translation is one of several tasks you can formulate as a sequence-to-sequence problem.
- Translation systems are commonly used for
    - translation between different language texts, (this notebook)
    - text-to-speech or speech-to-text.

## Load dependencies

In [ ]:
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import DataCollatorForSeq2Seq
import evaluate
import numpy as np
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import pipeline

## Config

In [ ]:
user_name = "amin-oj"
dataset_dict = {"path": "opus_books", "name": "en-fr"}
model_name = "google-t5/t5-small"
push_to_hub=True
train_bs = 16
eval_bs = 16
num_train_epochs = 3
task = "translation"
ckpt_name = f"{model_name.split("/")[-1]}-finetuned-{task}-opus-en-to-fr"
# ckpt_name = f"{model_name.split("/")[-1]}-finetuned-{task}-{dataset_dict["path"]}-{dataset_dict["split"]}"

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
books = load_dataset(**dataset_dict)
books = books["train"].train_test_split(test_size=0.2)
books["train"][0]

## Preprocess

In [ ]:
def preprocess_function(examples, prefix, tokenizer):
    inputs = [prefix + example["en"]
              for example in examples["translation"]]
    targets = [example["fr"] for
               example in examples["translation"]]
    model_inputs = tokenizer(inputs, text_target=targets,
                             max_length=128, truncation=True)
    return model_inputs

prefix = "translate English to French: "
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_books = books.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    fn_kwargs={"prefix": prefix, "tokenizer": tokenizer},
    remove_columns=books["train"].column_names
)

## Train

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)
# It's more efficient to *dynamically pad* the sentences to the longest length in a batch during collation,
# instead of padding the whole dataset to the maximum length.


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    metric = evaluate.load("sacrebleu")
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=ckpt_name,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    num_train_epochs=num_train_epochs,
    push_to_hub=push_to_hub,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=True,
    report_to = 'none' # to disable w&b
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_books["train"],
    eval_dataset=tokenized_books["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
if push_to_hub: trainer.push_to_hub()

## Inference

In [ ]:
text = "translate English to French: Legumes share resources with nitrogen-fixing bacteria."

### The simplest way

In [ ]:
translator = pipeline("translation_xx_to_yy", model=f"{user_name}/{ckpt_name}")
translator(text)

### More complex | More flexible

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(text, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained(f"{user_name}/{ckpt_name}")
outputs = model.generate(inputs, max_new_tokens=40, do_sample=True, top_k=30, top_p=0.95)
tokenizer.decode(outputs[0], skip_special_tokens=True)